# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
from smartcheck.logger_config import setup_logger

setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test tensorflow GPU config
import tensorflow as tf
print("✅ TF GPU:", tf.config.list_physical_devices("GPU"))

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np

# Pour les modèles et leur preprocessing
from sklearn.metrics import classification_report, confusion_matrix

# Pour la visualisation des performances
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Pour construire un réseau de neurone
from tensorflow.keras.layers import Dense, Dropout, Rescaling
from tensorflow.keras.layers import Input, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist


# 2. Loading and Data Enrichment

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print('Shape of X:', X_train.shape)
print('Shape of y:',y_train.shape)

In [ ]:
# Redimensionnement des données d'entraînement (X_train) et de test (X_test)
# (-1) permet de conserver le nombre d'images d'origine
# 28, 28, 1 spécifie la taille de chaque image (28x28 pixels avec 1 canal en niveaux de gris).
X_train = X_train.reshape((-1, 28, 28, 1))
X_test = X_test.reshape((-1, 28, 28, 1))
# Vérification des dimensions de l'ensemble d'entrainement
X_train.shape 

In [ ]:
# transformation en vecteur catégoriel binaire de la variable cible categorielle
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [ ]:
# Affichage aléatoire de 6 images
fig, axs = plt.subplots(2, 3, figsize=(14,9))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(np.arange(0, len(y_train)), size=6)):
    img = X_train[i]
    axs[j].axis('off')
    # Affichage de l'image en niveaux de gris
    axs[j].imshow(img, cmap='gray', interpolation='none')
    # Titre avec le label
    axs[j].set_title(f'Label: {str(np.argmax(y_train[i]))}')
plt.show()

In [ ]:
# image moyenne d'un label (moyenne par pixel de sur toutes les lignes d'un label donné)
fig, axs = plt.subplots(2, 5, figsize=(14,6))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
y_labels = np.argmax(y_train, axis=1)
for i in range(10):
    # Sélection des lignes de X_train correspondant au label i
    t = X_train[y_labels == i]
    # Calcul de l'image moyenne
    img = t.mean(axis=0).squeeze()   
    # Affichage de l'image dans le i+1-ème emplacement d'une grille de figures
    # à 2 lignes et 5 colonnes.    
    axs[i].imshow(img, cmap='gray', interpolation='None')

In [ ]:
# écart type d'un label (moyenne par pixel de sur toutes les lignes d'un label donné)
fig, axs = plt.subplots(2, 5, figsize=(14,6))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
y_labels = np.argmax(y_train, axis=1)
for i in range(10):
    # Sélection des lignes de X_train correspondant au label i
    t = X_train[y_labels == i]
    # Calcul de l'image moyenne
    img = t.std(axis=0).squeeze()     
    # Affichage de l'image dans le i+1-ème emplacement d'une grille de figures
    # à 2 lignes et 5 colonnes.    
    axs[i].imshow(img, cmap='gray_r', interpolation='None')

# 3. Deep learning

>Bonnes pratiques (computer vision)
> - couches convolutives (détection de features cachées : filtrage / bord / flou)
> - couches de pooling (réduction de dimension)

## 3.1 Modèle Tensor Flow Keras (Architecture LeNet)

#### Creation & Compilation

In [ ]:
# Définitions des dimension d'entrée et de sortie optimisées
shape_pixels = X_train.shape[1:]
num_classes = y_test.shape[1]

# Définition des layers selon une architecture LeNet
inputs = Input(
    shape=shape_pixels, 
    name="Input"
)
normalization_layer = Rescaling(1./255)
conv1_layer = Conv2D(
    filters=30, 
    kernel_size=(5, 5), 
    padding='valid',
    strides=(1,1),
    activation="relu", 
    name="Convolution2D_1"
)
pool1_layer = MaxPooling2D(
    pool_size=(2, 2),
    name='Pooling2D_1'
)
conv2_layer = Conv2D(
    filters=16, 
    kernel_size=(3, 3), 
    padding='valid',
    strides=(1,1),
    activation="relu", 
    name="Convolution2D_2"
)
pool2_layer = MaxPooling2D(
    pool_size=(2, 2),
    name='Pooling2D_2'
)
dropout_layer = Dropout(rate=0.2)
flatten_layer = Flatten()
dense1_layer = Dense(
    units=128,
    activation='relu',
    name='Dense_1'
)
output_layer = Dense(
    units=num_classes, 
    activation='softmax',
    name='Dense_output'
)

In [ ]:
# Instanciation du modèle par application successive des layers (fonctionnelle)
x = normalization_layer(inputs)
x = conv1_layer(x)
x = pool1_layer(x)
x = conv2_layer(x)
x = pool2_layer(x)
x = dropout_layer(x)
x = flatten_layer(x)
x = dense1_layer(x)
outputs = output_layer(x)
nn_tfkeras_func = Model(inputs=inputs, outputs=outputs)

In [ ]:
nn_tfkeras_func.compile(
    loss='categorical_crossentropy', # fonction de perte
    optimizer='adam',                # algorithme d'optimisation
    metrics=['accuracy'],            # métrique d'évaluation
)

In [ ]:
training_history = nn_tfkeras_func.fit(
    X_train, 
    y_train, 
    validation_split=0.2, 
    epochs=15, 
    batch_size=128, 
)

In [ ]:
train_acc = training_history.history['accuracy']
val_acc = training_history.history['val_accuracy']
train_loss = training_history.history['loss']
val_loss = training_history.history['val_loss']
fig, axs = plt.subplots(1, 2, figsize=(12,6))
axs[0].plot(train_loss, label='Loss (training)')
axs[0].plot(val_loss, label='Loss (validation)')
axs[0].set_title('Loss evolution per epoch')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[1].plot(train_acc, label='Accuracy (training)')
axs[1].plot(val_acc, label='Accuracy (validation)')
axs[1].set_title('Accuracy evolution per epoch')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('Accuracy')
axs[1].legend()
plt.show()

In [ ]:
# Prédictions du modèle : pour chaque échantillon, un vecteur de probabilités (1 par classe, grâce à softmax)
# Les classes sont ici codées de 0 à 9 (multi-classe)
y_test_prob = nn_tfkeras_func.predict(X_test)
y_test_pred_class = y_test_prob.argmax(axis=1)
y_test_class = y_test.argmax(axis=1)

In [ ]:
# evaluation du modèle (perte et accuracy)
print(nn_tfkeras_func.evaluate(X_test, y_test))

In [ ]:
# evaluation du modèle rapport de classification
print(classification_report(y_test_class, y_test_pred_class))

In [ ]:
# Calculer la matrice de confusion
cnf_matrix = confusion_matrix(y_test_class, y_test_pred_class, normalize='true')
# Tracer la heatmap de la matrice de confusion
plt.figure(figsize=(8, 6))
plt.title("Matrice de confusion")
sns.heatmap(cnf_matrix, cmap='Blues', annot=True, cbar=False, fmt=".2f")
plt.ylabel('Vrais labels')
plt.xlabel('Labels prédits')
plt.show()

In [ ]:
error_indexes = []
for i in range(len(y_test_prob)):
    if (y_test_pred_class[i] != y_test_class[i]):
        error_indexes += [i]
print(f"Nombre d'erreur du modèle [{len(error_indexes)}]")

# Affichage aléatoire des images sur lesquelles le modèle s'est trompé
(max_line, max_col) = (3, 3)
print(f"On en affiche aléatoirement {max_line}x{max_col}")
fig, axs = plt.subplots(max_line, max_col, figsize=(max_line*4,max_col*4))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(error_indexes, size=max_line*max_col)):
    img = X_test[i] 
    img = img.reshape(28, 28)
    axs[j].axis('off')
    # Affichage de l'image en niveaux de gris
    axs[j].imshow(img, cmap=plt.cm.binary, interpolation='none')
    # Titre avec le label
    axs[j].set_title(
        f'True Label: {str(y_test_class[i])}\n'
        f'Prediction: {str(y_test_pred_class[i])}\n'
        f'Confidence: {str(round(y_test_prob[i][y_test_pred_class[i]], 2))}'
    )